# DS2002 · Normalize and Join Nested JSON

**Lab — 2026-10-02 · Fall 2026**  

---

## Lab 06 — Normalize and Join Nested JSON

Two JSON payloads of the kind an API hands you. Flatten each, join them, and answer the questions.

Before you write anything, answer this for yourself: **what is one row of the table you are building?** Every argument you pass to `json_normalize` follows from that answer, and getting it wrong is the reason this lab takes people two hours instead of forty minutes.

One vendor in the orders is not in the vendor list. That is on purpose.

In [1]:
orders_json = [
  {'order':1,'vendor_id':'V-01','lines':[{'item':'Cheeseburger','qty':2},{'item':'Soda','qty':1}]},
  {'order':2,'vendor_id':'V-10','lines':[{'item':'Foam Finger','qty':1}]},
  {'order':3,'vendor_id':'V-07','lines':[{'item':'Hot Dog','qty':3}]},
]
vendors_json = [
  {'vendor_id':'V-01','name':'Hoos Burgers','zone':'A'},
  {'vendor_id':'V-10','name':'Cav Merch','zone':'A'},
]

**One row of my result is one:** one order placed, indicating the order number, vendor, etc.

### Q1 — Flatten the orders to one row per line item.

Keep `order` and `vendor_id` on every row.

*Expected: 4 rows, and the quantities should sum to 7.*

In [2]:
import pandas as pd
# TODO: pd.json_normalize(orders_json, record_path='lines', meta=[...])
# TODO: assert the row count and the quantity total
orders_normal = pd.json_normalize(orders_json, record_path="lines", meta=["order","vendor_id"])
assert len(orders_normal) == 4
assert orders_normal["qty"].sum() == 7

### Q2 — Join the vendor information on.

Left join, `indicator=True`. Report which vendor did not match and how many units are attached to it.

In [3]:
# TODO
vendors_normalized=pd.json_normalize(vendors_json)
orders_normal = pd.json_normalize(orders_json, record_path="lines", meta=["order","vendor_id"])
merged = orders_normal.merge(vendors_normalized, on="vendor_id", how="left",indicator=True)
print(merged)

           item  qty order vendor_id          name zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers    A       both
1          Soda    1     1      V-01  Hoos Burgers    A       both
2   Foam Finger    1     2      V-10     Cav Merch    A       both
3       Hot Dog    3     3      V-07           NaN  NaN  left_only


V-07 didn't match, and there are 3 quantities attatched to it.

### Q3 — Total quantity per vendor name.

Decide how the unmatched vendor appears in this table, and say why in a comment.

In [4]:
merged.loc[(merged["_merge"]=="left_only"),["name","zone"]] = ["unknown","unknown"]
print(merged)

           item  qty order vendor_id          name     zone     _merge
0  Cheeseburger    2     1      V-01  Hoos Burgers        A       both
1          Soda    1     1      V-01  Hoos Burgers        A       both
2   Foam Finger    1     2      V-10     Cav Merch        A       both
3       Hot Dog    3     3      V-07       unknown  unknown  left_only


The unmatched vendor appears in the table because the hot dog orders don't have a vendor corresponding to the vendor_id in the list of vendors.  I assigned this to be unknown in terms of both name and zone.

### Q4 — Which zone sold the most items?

Careful: if the unmatched vendor has no zone, your zone totals will not add up to 7. State what your total is and where the difference went.

In [5]:
# TODO
print(merged.groupby("zone")["qty"].sum())
print("total sum: ",merged["qty"].sum())

zone
A          4
unknown    3
Name: qty, dtype: int64
total sum:  7


The total quantity of orders is 7, with the A zone having a total of 4 and the unknown zone having a total of 3.

### Q5 — Handle a ragged payload.

A second batch arrives and it is not uniform: one order has no `lines` key at all, and one line item is missing its `qty`.

**TODO:** flatten what you can without crashing, and produce a frame where a missing quantity is distinguishable from a quantity of zero. Do **not** use `fillna(0)`.

In [6]:
ragged_json = [
    {'order': 4, 'vendor_id': 'V-01', 'lines': [{'item': 'Soda', 'qty': 2}]},
    {'order': 5, 'vendor_id': 'V-10'},                       # no lines at all
    {'order': 6, 'vendor_id': 'V-01', 'lines': [{'item': 'Fries'}]},  # no qty
]

# TODO
for i in ragged_json:  #check each dictionary in ragged_json
  if "lines" not in i:  #check if lines exists, if not, add an empty list
    i["lines"] = []
  else:
    for j in i["lines"]:
      if "qty" not in j: #check if each item in lines has a quantity as well. If not, assign quantity to be unknown
        j["qty"] = "unknown qty"
ragged_normal = pd.json_normalize(ragged_json, record_path="lines", meta=["order","vendor_id"])
print(ragged_normal)

    item          qty order vendor_id
0   Soda            2     4      V-01
1  Fries  unknown qty     6      V-01


### Q6 — Validate

In [7]:
lines = orders_normal
joined = merged
assert len(lines) == 4
assert lines['qty'].sum() == 7
assert len(joined) == len(lines), 'the vendor join changed the row count'
print('checks passed.')

checks passed.


### Write-up

**a)** Why is "one row per line item" the useful shape here? What question becomes easy that would be hard with one row per order?

**b)** In Q5, what is the practical difference between a missing quantity and a quantity of zero, and what breaks downstream if you conflate them?

a: one row per line item is useful because we can group the data by the contents of the order, like hot dogs and sodas.  It makes it easy to answer the question of how many of each type of good was ordered, which would be harder if it was one row per order.



b: a missing quantity indicates that you don't know how much of an item has been sold, whereas a quantity of zero indicates that you know none has been sold.  For the former, it's possible that you sold more than zero orders, you just don't know.  This makes data interpretation of sales downstream more difficult.